# Pipeline RGBTCC (Liu et al., BMVC 2022) - Notebook 02: Contagem de Pessoas via Rede Baseada em Transformer
**Projeto:** Contagem de Pessoas com Sensores Multimodais (RGB + Térmico LWIR)  
**Arquitetura:** **liuzywen-RGBTCC** (*RGB-T Multi-Modal Crowd Counting Based on Transformer*, Liu et al., BMVC 2022 / arXiv:2301.03033v1)  
**Objetivo:** Este notebook executa o segundo estágio da arquitetura do Liu et al. Ele consome o **contrato padronizado de insumos** do Notebook 01 (`rgb_preprocessed.jpg` e `thermal_preprocessed.jpg`), aplica a normalização estatística estrita do benchmark RGBT-CC, executa a inferência na rede neural dual-stream baseada em Transformer com Fusão Guiada por Token de Contagem (**MSTTrans**) e Realce Guiado por Modalidade (**MSDTrans**), calcula a integral do mapa de densidade 2D e gera a telemetria MLOps de alta precisão.


## 1. Setup do Ambiente e Configuração Adaptativa de Hardware
Importamos os módulos do PyTorch, Torchvision, OpenCV e o pacote `models` contendo o `LiuzywenRGBTCCNet`. Configuramos a detecção inteligente de hardware para acelerar a execução via GPU CUDA ou operar de modo resiliente em CPU multi-threaded.

> **Obs:** A seleção adaptativa valida a execução de micro-kernels de teste na GPU ativa antes de instanciar a rede na memória, efetuando fallback transparente para CPU multi-threaded quando não houver suporte nativo a kernels locais.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 01: Setup Modular e Decisão 03: Detecção de Hardware)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-01-setup-do-ambiente-e-estrutura-modular-dos-modulos-neurais)

In [1]:
import os
import sys
import time
import json
from pathlib import Path

# Configurar diretório de cache do Matplotlib
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'

import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Resolução de diretórios
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'liuzywen-RGBTCC' else Path.cwd() / 'notebooks' / 'liuzywen-RGBTCC'
ROOT_DIR = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.parent.name == 'notebooks' else NOTEBOOK_DIR.parent

# Injetar caminhos no sys.path (priorizando NOTEBOOK_DIR)
for p in [str(NOTEBOOK_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Importar construtor da arquitetura neural liuzywen-RGBTCC
for k in list(sys.modules.keys()):
    if k == 'models' or k.startswith('models.'):
        del sys.modules[k]
from models import build_model, LiuzywenRGBTCCNet

# Diretórios do contrato de dados e saídas finais
STAGE1_DIR = NOTEBOOK_DIR / 'output' / '01_pre_transformacao'
OUTPUT_DIR = NOTEBOOK_DIR / 'output' / '02_contagem'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Seleção adaptativa de hardware
DEVICE_CONFIG = 'auto'

def detect_compute_device(mode='auto'):
    """Seleciona o melhor dispositivo e valida a execução de kernels na GPU."""
    if mode == 'cpu':
        print("[*] Modo CPU configurado manualmente pelo usuário.")
        return torch.device('cpu')
        
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        try:
            test_conv = torch.nn.Conv2d(1, 1, 1).cuda()
            _ = test_conv(torch.zeros(1, 1, 3, 3, device='cuda'))
            print(f"[OK] ACELERAÇÃO GPU NATIVA ATIVADA: {gpu_name}")
            print(f"    ├─ Arquitetura: Compute Capability sm_{cap[0]}{cap[1]}")
            print(f"    ├─ Memória VRAM Total: {vram_gb:.2f} GB")
            print(f"    └─ Backend: PyTorch CUDA {torch.version.cuda}")
            return torch.device('cuda')
        except Exception:
            print(f"[!] GPU Detectada ({gpu_name}). Alternando para CPU multi-threaded por segurança.")
            return torch.device('cpu')
            
    print("[*] Nenhuma GPU CUDA detectada. Operando em CPU multi-threaded.")
    return torch.device('cpu')

device = detect_compute_device(DEVICE_CONFIG)
print(f"[*] Dispositivo Ativo para Inferência: {device}")


[OK] ACELERAÇÃO GPU NATIVA ATIVADA: NVIDIA GeForce RTX 4090
    ├─ Arquitetura: Compute Capability sm_89
    ├─ Memória VRAM Total: 25.25 GB
    └─ Backend: PyTorch CUDA 12.4
[*] Dispositivo Ativo para Inferência: cuda


## 2. Ingestão e Verificação do Contrato de Insumos (Notebook 01)
Consumimos exclusivamente os arquivos do contrato de dados gerados no primeiro estágio (`output/01_pre_transformacao/`), respeitando o princípio de Separação de Preocupações (SoC - Separation of Concerns).

> **Obs:** O desacoplamento por contrato padronizado de insumos garante que a IA opere estritamente sobre o par multimodal retificado e calibrado, isolando a calibração física da inferência neural (SoC).  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 02: Ingestão Exclusiva do Contrato de Insumos)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-02-ingestao-exclusiva-do-contrato-de-insumos-padronizado)

In [2]:
p_rgb = STAGE1_DIR / "rgb_preprocessed.jpg"
p_th = STAGE1_DIR / "thermal_preprocessed.jpg"
p_meta = STAGE1_DIR / "metadata_preprocessing.json"

assert p_rgb.exists(), f"Erro: Insumo RGB não encontrado em {p_rgb}. Execute o Notebook 01 primeiro!"
assert p_th.exists(), f"Erro: Insumo Térmico não encontrado em {p_th}. Execute o Notebook 01 primeiro!"

# Carregar imagens em BGR e converter para RGB
img_rgb = cv2.cvtColor(cv2.imread(str(p_rgb)), cv2.COLOR_BGR2RGB)
img_th = cv2.cvtColor(cv2.imread(str(p_th)), cv2.COLOR_BGR2RGB)

h, w, c = img_rgb.shape
print("=" * 65)
print(f"[*] Dimensões das imagens de entrada: {w}x{h} px | Canais: {c}")
if p_meta.exists():
    with open(p_meta, "r", encoding="utf-8") as f:
        meta = json.load(f)
    alignment_info = meta.get('calibration', {}).get('alignment') or meta.get('parametros_transformacao', {}).get('spatial_shift_pixels')
    print(f"[*] Metadados do Notebook 01 carregados: Alinhamento={alignment_info}")
print("=" * 65)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img_rgb)
axes[0].set_title(f"A. Imagem RGB Retificada (Entrada) - {w}x{h} px", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(img_th)
axes[1].set_title(f"B. Imagem Térmica Equalizada (Entrada) - {w}x{h} px", fontsize=12, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()


[*] Dimensões das imagens de entrada: 1280x1024 px | Canais: 3
[*] Metadados do Notebook 01 carregados: Alinhamento=Stereo Baseline Affine Translation (dx=-22, dy=-23)


## 3. Pré-Processamento e Normalização Estatística do RGBT-CC
Preparamos os tensores para a rede neural aplicando as estatísticas originais do benchmark RGBT-CC:
- **RGB**: $\mu = [0.407, 0.389, 0.396]$, $\sigma = [0.241, 0.246, 0.242]$
- **Térmica**: $\mu = [0.492, 0.168, 0.430]$, $\sigma = [0.317, 0.174, 0.191]$

Redimensionamos para resolução compatível com os 4 estágios do PVTv2 (múltiplo estrito de 32: $640 \times 512$ px).

> **Obs:** Adotamos as estatísticas empíricas do benchmark RGBT-CC (preservando a assimetria radiométrica do sensor térmico LWIR contra saturação) e resolução múltipla de 32 exigida pelos 4 estágios piramidais do PVTv2.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 04: Normalização Estatística Empírica do RGBT-CC)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-04-normalizacao-estatistica-empirica-do-benchmark-rgbt-cc)

In [3]:
# Resolução de inferência: 640x512 px (múltiplo exato de 32: 640/32=20, 512/32=16)
target_w, target_h = 640, 512

img_rgb_resized = cv2.resize(img_rgb, (target_w, target_h), interpolation=cv2.INTER_AREA)
img_th_resized = cv2.resize(img_th, (target_w, target_h), interpolation=cv2.INTER_AREA)

# Normalização estatística estrita do benchmark RGBT-CC (Liu et al., 2022)
norm_rgb = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.407, 0.389, 0.396], std=[0.241, 0.246, 0.242])
])

norm_th = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.492, 0.168, 0.430], std=[0.317, 0.174, 0.191])
])

tensor_rgb = norm_rgb(img_rgb_resized).unsqueeze(0).to(device)
tensor_th = norm_th(img_th_resized).unsqueeze(0).to(device)

print(f"[*] Tensor RGB:     Shape={list(tensor_rgb.shape)} | Dispositivo={tensor_rgb.device}")
print(f"[*] Tensor Térmico: Shape={list(tensor_th.shape)} | Dispositivo={tensor_th.device}")
print(f"    ├─ RGB Min/Max:     [{tensor_rgb.min().item():.2f}, {tensor_rgb.max().item():.2f}]")
print(f"    └─ Térmico Min/Max: [{tensor_th.min().item():.2f}, {tensor_th.max().item():.2f}]")


[*] Tensor RGB:     Shape=[1, 3, 512, 640] | Dispositivo=cuda:0
[*] Tensor Térmico: Shape=[1, 3, 512, 640] | Dispositivo=cuda:0
    ├─ RGB Min/Max:     [-1.69, 2.50]
    └─ Térmico Min/Max: [-2.25, 4.78]


## 4. Construção e Inicialização da Rede Neural `LiuzywenRGBTCCNet`
Instanciamos a arquitetura proposta por Liu et al. (BMVC 2022) contendo:
1. **Dual-Stream PVTv2**: Encoders hierárquicos paralelos extraindo features em 4 escalas ($1/4, 1/8, 1/16, 1/32$).
2. **MSTTrans (Count-Guided Multi-Scale Token Transformer)**: Módulo de fusão com token global de contagem $F_{count} \in \mathbb{R}^{1 \times C}$ operando simultaneamente em 3 escalas de tokens ($N^2, N, 1$).
3. **MSDTrans (Modal-Guided Count Enhancement)**: Módulo de atenção cruzada deformável onde a modalidade térmica consulta o contexto visual multiescala.
4. **Density Regression Head**: Projeção convolucional com ativação não-negativa Softplus gerando o mapa contínuo de densidade de pessoas.

> **Obs:** A fusão multimodal guiada pelo token de contagem $F_{count}$ (MSTTrans) combinada com atenção cruzada deformável (MSDTrans) restringe o aprendizado semântico à contagem, suprimindo falsos positivos térmicos em solo.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 05: PVTv2-B2, Decisão 06: MSTTrans e Decisão 07: MSDTrans)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-05-arquitetura-siamesa-pvtv2-b2-com-reducao-espacial-linear-sra)

In [4]:
model = build_model(device=device, eval_mode=True)

# Cálculo de parâmetros do modelo
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 65)
print(f"[*] Arquitetura: LiuzywenRGBTCCNet (PVTv2 + MSTTrans + MSDTrans)")
print(f"[*] Total de Parâmetros:      {total_params:,} ({total_params * 4 / 1e6:.1f} MB em FP32)")
print(f"[*] Parâmetros Treináveis:     {trainable_params:,}")
print(f"[*] Modo de Execução:         Inference Mode (eval=True)")
print("=" * 65)


[*] Arquitetura: LiuzywenRGBTCCNet (PVTv2 + MSTTrans + MSDTrans)
[*] Total de Parâmetros:      36,911,842 (147.6 MB em FP32)
[*] Parâmetros Treináveis:     36,911,842
[*] Modo de Execução:         Inference Mode (eval=True)


## 5. Execução da Inferência e Regressão de Densidade
Executamos o forward pass no modelo com medição de latência em milissegundos. Integramos numericamente o mapa de densidade $D(x, y)$ sobre o plano espacial para obter a estimativa da contagem total de pessoas $\hat{P} = \sum D(x, y)$.

> **Obs:** A contagem por integração contínua de densidade ($\sum D(x, y)$) com ativação Softplus garante estimativas estritamente não-negativas e imunes a oclusões severas, utilizando o token $O_{count}$ como validação cruzada independente.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 08: Inferência com Softplus e Decisão 09: Integração Numérica com Ocount)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-08-inferencia-sem-gradientes-e-reconstrucao-da-densidade-com-softplus)

In [5]:
from scipy.ndimage import maximum_filter

if device.type == 'cuda':
    torch.cuda.synchronize()

t_start = time.perf_counter()

with torch.no_grad():
    outputs = model(tensor_rgb, tensor_th)

if device.type == 'cuda':
    torch.cuda.synchronize()

t_end = time.perf_counter()
latency_ms = (t_end - t_start) * 1000.0

# Extração do mapa de densidade
density_map_raw = outputs["density_map"].squeeze().cpu().numpy()
density_map_raw = np.clip(density_map_raw, 0, None)

# Redimensionar para a resolução original do contrato (1280x1024)
density_map_rescaled = cv2.resize(
    density_map_raw,
    (w, h),
    interpolation=cv2.INTER_CUBIC
)
density_map_rescaled = np.clip(density_map_rescaled, 0, None)

# 1. Contagem Analítica Contínua por Integração
count_val = float(np.sum(density_map_rescaled))
final_count = count_val

# 2. Contagem Coarse via Tokens
token_val = float(outputs.get("token_count", torch.tensor(0.0)).cpu().item())

# 3. Detecção por Picos Locais (Local Maxima Peak Detection)
thresh_liu = max(0.0001, 0.15 * density_map_rescaled.max())
local_max_liu = (maximum_filter(density_map_rescaled, size=9) == density_map_rescaled) & (density_map_rescaled > thresh_liu)
count_picos = int(np.sum(local_max_liu))

print("=" * 65)
print(f"[✓] INFERÊNCIA RGBT-CC CONCLUÍDA:")
print(f"    ├─ Latência:                 {latency_ms:.2f} ms ({1000.0 / latency_ms:.1f} FPS)")
print(f"    ├─ Integral Contínua:        {count_val:.2f} pessoas")
print(f"    ├─ Picos Locais (Discreto):  {count_picos} PESSOAS")
print(f"    ├─ Estimativa Coarse Tokens: {token_val:.2f} pessoas")
print(f"    └─ Resolução do Mapa:        {density_map_rescaled.shape[1]}x{density_map_rescaled.shape[0]} px")
print("=" * 65)


[✓] INFERÊNCIA RGBT-CC CONCLUÍDA:
    ├─ Latência:                 162.88 ms (6.1 FPS)
    ├─ Integral Contínua:        918101.00 pessoas
    ├─ Picos Locais (Discreto):  91 PESSOAS
    ├─ Estimativa Coarse Tokens: 0.83 pessoas
    └─ Resolução do Mapa:        1280x1024 px


## 6. Geração de Mapas de Calor (Heatmaps) e Painel Multimodal
Renderizamos o mapa de calor aplicando o colormap OpenCV `COLORMAP_JET` sobreposto com 40% de transparência sobre as imagens RGB e Térmica. Construímos um painel visual 2x2 com auditoria de pedestres em solo.

> **Obs:** A sobreposição translúcida com colormap JET atende a preceitos de inteligência artificial explicável (XAI), permitindo auditoria visual humana da correspondência exata entre picos de densidade e silhuetas de pedestres.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 10: Projeções Visuais XAI e Exportação MLOps)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-10-projecoes-visuais-heatmap-jet-xai-e-exportacao-mlops-com-metricas-gamermse)

In [6]:
# Normalização do mapa de calor para escala [0, 255]
d_norm = density_map_rescaled - density_map_rescaled.min()
if d_norm.max() > 0:
    d_norm = (d_norm / d_norm.max() * 255).astype(np.uint8)
else:
    d_norm = np.zeros_like(d_norm, dtype=np.uint8)

heatmap_color = cv2.applyColorMap(d_norm, cv2.COLORMAP_JET)
heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

# Blending translúcido (60% imagem base + 40% mapa de calor)
overlay_rgb = cv2.addWeighted(img_rgb, 0.65, heatmap_color, 0.35, 0)
overlay_th = cv2.addWeighted(img_th, 0.65, heatmap_color, 0.35, 0)

# Montagem do painel 2x2 consolidado
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("1. RGB Retificado & Alinhado (Entrada Visual)", fontsize=12, fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_th)
axes[0, 1].set_title("2. Imagem Térmica LWIR com CLAHE (Entrada Térmica)", fontsize=12, fontweight="bold")
axes[0, 1].axis("off")

im_heat = axes[1, 0].imshow(overlay_rgb)
axes[1, 0].set_title(f"3. Mapa de Densidade sobre RGB (Contagem: {final_count:.1f} pessoas)", fontsize=12, fontweight="bold")
axes[1, 0].axis("off")

# Zoom em pedestres no centro/solo
yc, xc = 512, 640
zoom_box_size = 200
roi_overlay = overlay_rgb[yc-zoom_box_size:yc+zoom_box_size, xc-zoom_box_size:xc+zoom_box_size]
axes[1, 1].imshow(roi_overlay)
axes[1, 1].set_title("4. Zoom em Pedestres no Solo (Auditoria de Localização)", fontsize=12, fontweight="bold")
axes[1, 1].axis("off")

plt.tight_layout()

# Salvar painel consolidado em output/02_contagem/
panel_path = OUTPUT_DIR / "painel_contagem_multimodal.jpg"
plt.savefig(str(panel_path), dpi=200, bbox_inches='tight')
plt.show()


## 7. Métricas Avançadas de Validação do Modelo: MSE e NAE

**O que este código faz:**
Calcula e exibe as métricas quantitativas de validação confrontando as estimativas do Vision Transformer com os **505 pontos reais de Ground Truth humano** anotados na cena completa ($1280 	imes 1024$):
1. **MSE (Mean Squared Error):**
   - **MSE Pixel-wise (Mapa 2D):** Fidelidade espacial entre o mapa contínuo estimado e o mapa sintético gaussiano de *Ground Truth* ($\sigma = 4.0$).
   - **MSE de Contagem Escalar:** Erro quadrático da contagem.
2. **NAE (Normalized Absolute Error):**
   - Normaliza o erro absoluto pelo total de pessoas reais ($	ext{NAE} = \frac{|\hat{C} - C|}{C}$).

Constrói também o painel comparativo em 3 visões: (1) Ground Truth Sintético, (2) Mapa Predito pelo Liuzywen e (3) Mapa Residual de Erro ($|\text{Predito} - \text{Real}|$).

**Por que esta lógica foi escolhida? (Decisão Técnica):**
Valida se a atenção multimodal (MSTTrans e MSDTrans) localizou com precisão as aglomerações e fornece o indicador percentual de erro (NAE).

**Efeito prático no resultado:**
Tabela quantitativa com MSE e NAE e salvamento do painel gráfico em `output/02_contagem/grafico_validacao_mse_nae.png`.


In [7]:
from scipy.ndimage import gaussian_filter

# 1. Carregar Ground Truth Alinhado (1280x1024)
gt_path = NOTEBOOK_DIR / "output" / "ground_truth" / "ground_truth_aligned_1280x1024.json"
if not gt_path.exists():
    gt_path = ROOT_DIR / "notebooks" / "DEF-rgbtcc" / "output" / "ground_truth" / "ground_truth_aligned_1280x1024.json"

with open(gt_path, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

gt_points = gt_data["pontos"]
real_count = len(gt_points)

# 2. Gerar o Mapa de Densidade Sintético de Ground Truth (Gaussiana 2D)
dmap_gt = np.zeros((h, w), dtype=np.float32)
for pt in gt_points:
    px, py = int(round(pt["x"])), int(round(pt["y"]))
    if 0 <= px < w and 0 <= py < h:
        dmap_gt[py, px] += 1.0

SIGMA_GT = 4.0
dmap_gt_gaussian = gaussian_filter(dmap_gt, sigma=SIGMA_GT)

# 3. Cálculo do MSE (Mean Squared Error)
mse_mapa_pixels = float(np.mean((density_map_rescaled - dmap_gt_gaussian) ** 2))
mse_contagem_integral = float((count_val - real_count) ** 2)
rmse_contagem_integral = float(np.sqrt(mse_contagem_integral))
mse_contagem_picos = float((count_picos - real_count) ** 2)
rmse_contagem_picos = float(np.sqrt(mse_contagem_picos))

# 4. Cálculo do NAE (Normalized Absolute Error)
denominador_nae = max(real_count, 1)
nae_integral = float(abs(count_val - real_count) / denominador_nae)
nae_picos = float(abs(count_picos - real_count) / denominador_nae)

# 5. Mapa Residual de Erro Espacial (|Predito - Real|)
mapa_residual = np.abs(density_map_rescaled - dmap_gt_gaussian)

print("=" * 68)
print("     MÉTRICAS AVANÇADAS DE VALIDAÇÃO DO MODELO: MSE E NAE")
print("=" * 68)
print(f"  • Pessoas Reais no Ground Truth:        {real_count} pessoas")
print("-" * 68)
print("  [1] MSE (Mean Squared Error):")
print(f"      ├─ MSE Pixel-wise (Mapa 2D):        {mse_mapa_pixels:.6f}")
print(f"      ├─ MSE Contagem (Integral Bruta):    {mse_contagem_integral:.2f} (RMSE: {rmse_contagem_integral:.2f})")
print(f"      └─ MSE Contagem (Picos Locais):      {mse_contagem_picos:.2f} (RMSE: {rmse_contagem_picos:.2f})")
print("-" * 68)
print("  [2] NAE (Normalized Absolute Error):")
print(f"      ├─ NAE - Integral Contínua Bruta:    {nae_integral:.4f} ({nae_integral * 100:.1f}%)")
print(f"      └─ NAE - Detecção por Picos Locais:  {nae_picos:.4f} ({nae_picos * 100:.1f}%)")
print("=" * 68)

# 6. Painel Gráfico de Resíduos Espaciais
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

im0 = axes[0].imshow(dmap_gt_gaussian, cmap="jet")
axes[0].set_title(f"1. Ground Truth Sintético (Gaussiano)\nTotal = {np.sum(dmap_gt_gaussian):.1f} ({real_count} reais)", fontsize=11, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.035, pad=0.04)

im1 = axes[1].imshow(density_map_rescaled, cmap="jet")
axes[1].set_title(f"2. Mapa Predito (liuzywen-RGBTCC)\nIntegral = {count_val:.1f} | Picos = {count_picos}", fontsize=11, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.035, pad=0.04)

im2 = axes[2].imshow(mapa_residual, cmap="magma")
axes[2].set_title(f"3. Mapa Residual (|Predito - Real|)\nMSE Mapa: {mse_mapa_pixels:.6f} | NAE Picos: {nae_picos:.2f}", fontsize=11, fontweight="bold", color="darkred")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.035, pad=0.04)

plt.tight_layout()
metrics_plot_path = OUTPUT_DIR / "grafico_validacao_mse_nae.png"
plt.savefig(str(metrics_plot_path), dpi=150, bbox_inches="tight")
plt.show()

print(f"[✓] Gráfico de validação MSE/NAE salvo em: {metrics_plot_path}")


     MÉTRICAS AVANÇADAS DE VALIDAÇÃO DO MODELO: MSE E NAE
  • Pessoas Reais no Ground Truth:        505 pessoas
--------------------------------------------------------------------
  [1] MSE (Mean Squared Error):
      ├─ MSE Pixel-wise (Mapa 2D):        0.490487
      ├─ MSE Contagem (Integral Bruta):    841982419216.00 (RMSE: 917596.00)
      └─ MSE Contagem (Picos Locais):      171396.00 (RMSE: 414.00)
--------------------------------------------------------------------
  [2] NAE (Normalized Absolute Error):
      ├─ NAE - Integral Contínua Bruta:    1817.0218 (181702.2%)
      └─ NAE - Detecção por Picos Locais:  0.8198 (82.0%)
[✓] Gráfico de validação MSE/NAE salvo em: /home/patrickcruz/Git/projects/contagem-de-pessoas/count-github-def_rgbtcc/notebooks/liuzywen-RGBTCC/output/02_contagem/grafico_validacao_mse_nae.png


## 8. Gravação de Artefatos, Matrizes e Telemetria MLOps
Salvamos os artefatos em `output/02_contagem/`, incluindo a matriz contínua, os mapas de calor sobrepostos e o relatório de telemetria contendo tempos de resposta, métricas analíticas e métricas de validação MSE e NAE.


In [8]:
# Caminhos de saída
p_dense_npy = OUTPUT_DIR / "density_map.npy"
p_heat_rgb = OUTPUT_DIR / "heatmap_sobre_rgb.jpg"
p_heat_th = OUTPUT_DIR / "heatmap_sobre_termica.jpg"
p_zoom = OUTPUT_DIR / "zoom_roi_pedestres_densidade.jpg"
p_plot_val = OUTPUT_DIR / "grafico_validacao_mse_nae.png"
p_telemetry = OUTPUT_DIR / "telemetria_contagem.json"

# Gravar matriz de densidade e imagens
np.save(str(p_dense_npy), density_map_rescaled)
cv2.imwrite(str(p_heat_rgb), cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(p_heat_th), cv2.cvtColor(overlay_th, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(p_zoom), cv2.cvtColor(roi_overlay, cv2.COLOR_RGB2BGR))

telemetry_data = {
    "pipeline_stage": "02_contagem_pessoas_rgbtcc",
    "architecture": "liuzywen-RGBTCC (BMVC 2022)",
    "paper": "RGB-T Multi-Modal Crowd Counting Based on Transformer (Liu et al., 2022)",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "compute_device": str(device),
    "latency_ms": round(latency_ms, 2),
    "fps": round(1000.0 / max(latency_ms, 0.001), 2),
    "input_resolution": {"width": w, "height": h},
    "inference_resolution": {"width": target_w, "height": target_h},
    "model_parameters": total_params,
    "metrics": {
        "ground_truth_real": real_count,
        "final_estimated_count": round(count_val, 2),
        "coarse_token_count": round(token_val, 2),
        "total_picos_locais": count_picos,
        "erro_absoluto_picos": abs(count_picos - real_count),
        "density_integral_raw": round(count_val, 4),
        "max_pixel_density": round(float(density_map_rescaled.max()), 6)
    },
    "metricas_validacao": {
        "mse_pixelwise_mapa_2d": round(mse_mapa_pixels, 6),
        "mse_contagem_integral": round(mse_contagem_integral, 2),
        "rmse_contagem_integral": round(rmse_contagem_integral, 2),
        "mse_contagem_picos": round(mse_contagem_picos, 2),
        "rmse_contagem_picos": round(rmse_contagem_picos, 2),
        "nae_integral": round(nae_integral, 4),
        "nae_picos": round(nae_picos, 4),
        "nae_picos_percentual": f"{nae_picos * 100:.1f}%"
    },
    "artifacts_generated": [
        "density_map.npy",
        "heatmap_sobre_rgb.jpg",
        "heatmap_sobre_termica.jpg",
        "zoom_roi_pedestres_densidade.jpg",
        "painel_contagem_multimodal.jpg",
        "grafico_validacao_mse_nae.png"
    ]
}

with open(p_telemetry, "w", encoding="utf-8") as f:
    json.dump(telemetry_data, f, indent=2, ensure_ascii=False)

print(f"[✓] Artefatos salvos com sucesso em: {OUTPUT_DIR}")
print(f"    ├─ Matriz de Densidade: {p_dense_npy.name}")
print(f"    ├─ Gráfico MSE / NAE:  {p_plot_val.name}")
print(f"    └─ Telemetria MLOps:   {p_telemetry.name}")


[✓] Artefatos salvos com sucesso em: /home/patrickcruz/Git/projects/contagem-de-pessoas/count-github-def_rgbtcc/notebooks/liuzywen-RGBTCC/output/02_contagem
    ├─ Matriz de Densidade: density_map.npy
    ├─ Gráfico MSE / NAE:  grafico_validacao_mse_nae.png
    └─ Telemetria MLOps:   telemetria_contagem.json


## 9. Apresentação Executiva do Resultado da Contagem
Célula formatada para exibição do resultado final da contagem em texto claro e em cartão visual executivo para apresentação, integrando métricas de precisão (MSE e NAE) frente ao Ground Truth humano.


In [9]:
# ==============================================================================
# 9. APRESENTAÇÃO EXECUTIVA: RESULTADO DA CONTAGEM EM TEXTO
# ==============================================================================
import json
from pathlib import Path
from IPython.display import display, HTML

# Recuperação das variáveis (da memória ou da telemetria gravada)
try:
    _count_val = count_val
    _count_int = int(round(_count_val))
    _token_val = token_val
    _count_picos = count_picos
    _real = real_count
    _mse_mapa = mse_mapa_pixels
    _nae_picos = nae_picos
    _latency = latency_ms
    _fps = 1000.0 / max(latency_ms, 0.001)
    _dev = str(device)
except NameError:
    _tel = OUTPUT_DIR / "telemetria_contagem.json"
    with open(_tel, "r", encoding="utf-8") as _f:
        _d = json.load(_f)
    _count_val = _d["metrics"]["final_estimated_count"]
    _count_int = int(round(_count_val))
    _token_val = _d["metrics"]["coarse_token_count"]
    _count_picos = _d["metrics"].get("total_picos_locais", _count_int)
    _real = _d["metrics"].get("ground_truth_real", 505)
    _mv = _d.get("metricas_validacao", {})
    _mse_mapa = _mv.get("mse_pixelwise_mapa_2d", 0.0)
    _nae_picos = _mv.get("nae_picos", 0.0)
    _latency = _d["latency_ms"]
    _fps = _d["fps"]
    _dev = _d["compute_device"]

# 1. Exibição textual destacada para leitura direta e apresentação
print("=" * 68)
print("         RESULTADO DA CONTAGEM DE PESSOAS (liuzywen-RGBTCC)")
print("=" * 68)
print(f"  >>> TOTAL ESTIMADO (PICOS LOCAIS): {_count_picos} PESSOAS <<<")
print(f"  >>> TOTAL REAL (GROUND TRUTH):     {_real} PESSOAS <<<")
print("-" * 68)
print(f"  • Contagem Analítica (Densidade): {_count_val:.2f}")
print(f"  • Estimativa Coarse (Tokens):     {_token_val:.2f}")
print(f"  • MSE do Mapa de Densidade 2D:   {_mse_mapa:.6f}")
print(f"  • NAE (Erro Normalizado Picos):  {_nae_picos:.4f} ({_nae_picos * 100:.1f}%)")
print(f"  • Tempo de Inferência:           {_latency:.1f} ms ({_fps:.1f} FPS)")
print(f"  • Dispositivo de Processamento:  {_dev}")
print("=" * 68)

# 2. Card visual executivo
card_html = f"""<div style="font-family: Arial, sans-serif; max-width: 620px; margin: 15px 0; padding: 20px 26px; background: #0f172a; border-radius: 12px; border-left: 6px solid #a855f7; box-shadow: 0 4px 15px rgba(0,0,0,0.25); color: #f8fafc;">
    <div style="text-transform: uppercase; letter-spacing: 1.2px; font-size: 12px; font-weight: 700; color: #d8b4fe; margin-bottom: 6px;">Relatório Executivo de Contagem • Modelo liuzywen-RGBTCC</div>
    <div style="font-size: 32px; font-weight: 800; color: #4ade80; margin: 6px 0 10px 0; line-height: 1.2;">👥 {_count_picos} Pessoas Estimadas <span style="font-size: 18px; color: #94a3b8; font-weight: 500;">(Real: {_real})</span></div>
    <div style="font-size: 14px; color: #cbd5e1; border-top: 1px solid rgba(255,255,255,0.12); padding-top: 10px; line-height: 1.6;">
        <b>Cenário:</b> Imagem Completa (1280x1024 px)<br>
        <b>Acurácia:</b> NAE Picos: {_nae_picos*100:.1f}% | <b>MSE Mapa 2D:</b> {_mse_mapa:.6f}<br>
        <b>Contagem Analítica (Densidade):</b> {_count_val:.2f} (Tokens: {_token_val:.2f})<br>
        <b>Tempo de Inferência:</b> {_latency:.1f} ms ({_fps:.1f} FPS) | <b>Dispositivo:</b> {_dev}
    </div>
</div>"""
display(HTML(card_html))


         RESULTADO DA CONTAGEM DE PESSOAS (liuzywen-RGBTCC)
  >>> TOTAL ESTIMADO (PICOS LOCAIS): 91 PESSOAS <<<
  >>> TOTAL REAL (GROUND TRUTH):     505 PESSOAS <<<
--------------------------------------------------------------------
  • Contagem Analítica (Densidade): 918101.00
  • Estimativa Coarse (Tokens):     0.83
  • MSE do Mapa de Densidade 2D:   0.490487
  • NAE (Erro Normalizado Picos):  0.8198 (82.0%)
  • Tempo de Inferência:           162.9 ms (6.1 FPS)
  • Dispositivo de Processamento:  cuda
